In [ ]:
import cv2
import mediapipe as mp
import time
import numpy as np
import torch

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

cap = cv2.VideoCapture(0)

sequence = []

if cap.isOpened():
    print("카메라가 성공적으로 켜졌습니다.", flush=True)
    print("종료하려면 카메라 창 클릭 후 q를 누르세요.", flush=True)
    time.sleep(2)
else:
    print("카메라 연결 실패")

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        print("카메라 화면을 불러올 수 없습니다.")
        break

    frame = cv2.flip(frame, 1)

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(img_rgb)
    frame_keypoints = []

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            for lm in hand_landmarks.landmark:
                frame_keypoints.append(lm.x)
                frame_keypoints.append(lm.y)
                frame_keypoints.append(1)

    while len(frame_keypoints) < 126:
        frame_keypoints.append(0)

    frame_keypoints = frame_keypoints[:126]

    extra_features = [0, 0, 0, 0, 0, 0]
    frame_keypoints = frame_keypoints + extra_features
    
    sequence.append(frame_keypoints)
    if len(sequence) == 30:

        sequence_array = np.array(sequence)
        sequence_tensor = torch.tensor(sequence_array, dtype=torch.float32)
        sequence_tensor = sequence_tensor.unsqueeze(0)

        print("30프레임 수집 완료")
        print("sequence shape:", sequence_array.shape)
        print("tensor shape:", sequence_tensor.shape)

        sequence = []


    print("좌표 개수:", len(frame_keypoints))
    print(frame_keypoints[:9])
    if results.multi_hand_landmarks:

        for hand_landmarks in results.multi_hand_landmarks:

            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=4),
                mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2)
            )

    cv2.imshow("Su-eo Project: Hand Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()